### Context and State

In [44]:
from langchain.agents import AgentState
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command
from langchain.messages import ToolMessage
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver
from pprint import pprint

Globals

In [45]:
model = f"bedrock:us.anthropic.claude-opus-4-6-v1"

Creating State

In [46]:
class CustomState(AgentState):
    fav_color: str

Tool for State

In [47]:
@tool
def update_color(fav_color: str, runtime: ToolRuntime) -> Command:
    """Update Color

    Args:
        fav_color (str): favourite color
        runtime (ToolRuntime): agent runtime

    Returns:
        Command: command to be returned
    """
    
    
    return Command(
        update={
            "fav_color": fav_color,
            "messages": [ToolMessage("Updated color", tool_call_id=runtime.tool_call_id)]
        }
    )

Agent

In [48]:
agent = create_agent(
    model=model, 
    tools=[update_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

Updating State Through The Prompt

In [49]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'fav_color': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='e46e2c9a-576d-4808-952d-f46d47eb57f3'),
              AIMessage(content="\n\nI'll update your favourite colour to green right away!", additional_kwargs={'usage': {'prompt_tokens': 610, 'completion_tokens': 69, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 679}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1'}, response_metadata={'usage': {'prompt_tokens': 610, 'completion_tokens': 69, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 679}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1', 'model_provider': 'bedrock', 'model_name': 'us.anthropic.claude-opus-4-6-v1'}, id='lc_run--019cde1e-55c6-7a11-a71a-295d6ba70e54-0', tool_calls=[{'name': 'update_color', 'args': {'fav_color': 'green'}, 'id': 'toolu_bdrk_01U7CtXcerFAxZcDWJ6uL2bZ', 'type': 'tool_c

Changing State at Agent Invocation

In [50]:
response = agent.invoke(
    { 
        "messages": [HumanMessage(content="Hello, how are you?")],
        "fav_color": "red"
    },
    {"configurable": {"thread_id": "10"}}
)

pprint(response)

{'fav_color': 'red',
 'messages': [HumanMessage(content='Hello, how are you?', additional_kwargs={}, response_metadata={}, id='26d92307-611b-4f7e-852c-179ef7d74de4'),
              AIMessage(content="Hello! I'm doing great, thank you for asking! 😊 How are you doing today? Is there anything I can help you with?", additional_kwargs={'usage': {'prompt_tokens': 611, 'completion_tokens': 34, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 645}, 'stop_reason': 'end_turn', 'model_id': 'us.anthropic.claude-opus-4-6-v1'}, response_metadata={'usage': {'prompt_tokens': 611, 'completion_tokens': 34, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 645}, 'stop_reason': 'end_turn', 'model_id': 'us.anthropic.claude-opus-4-6-v1', 'model_provider': 'bedrock', 'model_name': 'us.anthropic.claude-opus-4-6-v1'}, id='lc_run--019cde1e-a902-7ab0-82bc-15a63e9307ad-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 611, 'output_toke

Reading State

In [51]:
@tool
def read_color(runtime: ToolRuntime) -> str:
    """read the favourite color of the user from the state"""
    try:
        return runtime.state["fav_color"]
    except KeyError:
        return "no favourite colour found in state"

Agent

In [52]:
agent = create_agent(
    model=model, 
    tools=[update_color, read_color],
    checkpointer=InMemorySaver(),
    state_schema=CustomState
)

Updating State and Reading from State

In [53]:
response = agent.invoke(
    { "messages": [HumanMessage(content="My favourite colour is green")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'fav_color': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='040f048b-64b1-41fe-af95-222449a5f5e3'),
              AIMessage(content="\n\nI'll update your favourite colour to green right away!", additional_kwargs={'usage': {'prompt_tokens': 654, 'completion_tokens': 69, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 723}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1'}, response_metadata={'usage': {'prompt_tokens': 654, 'completion_tokens': 69, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 723}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1', 'model_provider': 'bedrock', 'model_name': 'us.anthropic.claude-opus-4-6-v1'}, id='lc_run--019cde1e-b69e-7eb2-a5a3-2fc36e87da93-0', tool_calls=[{'name': 'update_color', 'args': {'fav_color': 'green'}, 'id': 'toolu_bdrk_01EDykF9n3hRy2Vjp2QCv5kg', 'type': 'tool_c

In [54]:
response = agent.invoke(
    { "messages": [HumanMessage(content="What's my favourite colour?")]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'fav_color': 'green',
 'messages': [HumanMessage(content='My favourite colour is green', additional_kwargs={}, response_metadata={}, id='040f048b-64b1-41fe-af95-222449a5f5e3'),
              AIMessage(content="\n\nI'll update your favourite colour to green right away!", additional_kwargs={'usage': {'prompt_tokens': 654, 'completion_tokens': 69, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 723}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1'}, response_metadata={'usage': {'prompt_tokens': 654, 'completion_tokens': 69, 'cache_read_input_tokens': 0, 'cache_write_input_tokens': 0, 'total_tokens': 723}, 'stop_reason': 'tool_use', 'model_id': 'us.anthropic.claude-opus-4-6-v1', 'model_provider': 'bedrock', 'model_name': 'us.anthropic.claude-opus-4-6-v1'}, id='lc_run--019cde1e-b69e-7eb2-a5a3-2fc36e87da93-0', tool_calls=[{'name': 'update_color', 'args': {'fav_color': 'green'}, 'id': 'toolu_bdrk_01EDykF9n3hRy2Vjp2QCv5kg', 'type': 'tool_c